In [8]:
!pip install python-docx

In [9]:
from docx import Document

def extract_resume_text(file_path):
    ext = os.path.splitext(file_path)[1].lower()
    text = ""
    
    try:
        if ext == '.pdf':
            with pdfplumber.open(file_path) as pdf:
                for page in pdf.pages:
                    page_text = page.extract_text()
                    if page_text:
                        text += page_text + "\n"
        elif ext == '.docx':
            doc = Document(file_path)
            for para in doc.paragraphs:
                text += para.text + "\n"
        else:
            with open(file_path, 'r', encoding='utf-8', errors='ignore') as f:
                text = f.read()
        return text.strip()
    except Exception as e:
        print(f"Error reading {file_path}: {e}")
        return ""

In [10]:
# FUTURE_ML_03 – Resume Screening & Candidate Ranking System
# Future Interns Machine Learning Internship – Task 3
# Author: Narkeesbanus | Chennai, Tamil Nadu | February 2026

import os
import re
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import pdfplumber
from docx import Document   # Now supporting .docx
import spacy
import nltk
from nltk.corpus import stopwords
from datetime import datetime

# ── Initial Setup ──────────────────────────────────────────────────
nltk.download('stopwords', quiet=True)
stop_words = set(stopwords.words('english'))
nlp = spacy.load("en_core_web_sm")

print("All dependencies loaded successfully.")

# ── 1. Resume Text Extractor (PDF + DOCX + text fallback) ──────────
def extract_resume_text(file_path):
    ext = os.path.splitext(file_path)[1].lower()
    text = ""
    
    try:
        if ext == '.pdf':
            with pdfplumber.open(file_path) as pdf:
                for page in pdf.pages:
                    page_text = page.extract_text()
                    if page_text:
                        text += page_text + "\n"
                        
        elif ext == '.docx':
            doc = Document(file_path)
            for para in doc.paragraphs:
                if para.text.strip():
                    text += para.text + "\n"
                    
        else:  # .txt or any readable file
            with open(file_path, 'r', encoding='utf-8', errors='ignore') as f:
                text = f.read()
                
        return text.strip()
    except Exception as e:
        print(f"Error reading {file_path}: {e}")
        return ""

# ── 2. Advanced Feature Extraction ─────────────────────────────────
def extract_features(text):
    if not text:
        return {
            'skills': [],
            'years_experience': 0,
            'education_level': 'Unknown',
            'raw_preview': ""
        }
    
    doc = nlp(text.lower())
    skills = set()
    
    # Named entities
    for ent in doc.ents:
        if ent.label_ in ['ORG', 'GPE', 'DATE', 'NORP', 'PRODUCT']:
            skills.add(ent.text.strip())
    
    # Expanded regex for skills (covers financial + design + ML terms)
    skill_patterns = r'(python|java|c\+\+|sql|machine learning|deep learning|nlp|tensorflow|pytorch|keras|scikit-learn|pandas|numpy|aws|azure|git|docker|kubernetes|react|javascript|html|css|data science|ai|ml|big data|hadoop|spark|tableau|power bi|excel|communication|leadership|teamwork|agile|devops|financial advisor|investment|portfolio management|estate planning|business administration|graphic design|adobe creative suite|illustrator|photoshop|indesign|dreamweaver|figma|framer|procreate|ms office|microsoft office|roi|asset management|client satisfaction|brand awareness|web design|senior|junior|specialist|advisor|wells fargo|suntrust|maverick capital|experion|stepping stone|redfin)'
    found = re.findall(skill_patterns, text.lower())
    skills.update(found)
    
    # Estimate years of experience
    years = 0
    date_matches = re.findall(r'\b(20\d{2})\b', text)
    if date_matches:
        try:
            earliest = min(int(y) for y in date_matches if int(y) < datetime.now().year)
            years = datetime.now().year - earliest
            years = max(0, min(years, 30))
        except:
            years = 0
    
    # Education level
    edu_keywords = ['phd', 'doctorate', 'master', 'mba', 'bachelor', 'bs', 'ba', 'b.tech', 'b.e', 'degree']
    edu_level = 'Unknown'
    for kw in edu_keywords:
        if kw in text.lower():
            edu_level = kw.capitalize()
            break
    
    return {
        'skills': sorted(list(set(skills))),
        'years_experience': years,
        'education_level': edu_level,
        'raw_preview': text[:400] + "..." if len(text) > 400 else text
    }

# ── 3. Clean Text for Similarity ───────────────────────────────────
def clean_text(text):
    if not text:
        return ""
    text = re.sub(r'[^a-zA-Z\s]', '', text.lower())
    words = [w for w in text.split() if w not in stop_words and len(w) > 2]
    return ' '.join(words)

# ── 4. Main Processing ─────────────────────────────────────────────
resume_folder = r"C:\Users\NARKEES SALEEM\resumes"  # Your confirmed path

job_desc_path = "sample_job_description.txt"

if not os.path.exists(resume_folder):
    print(f"Folder not found: {resume_folder}")
else:
    print(f"Scanning: {resume_folder}")

# Create sample job desc if missing
if not os.path.exists(job_desc_path):
    default_job = """
    Hiring versatile professionals:

    Machine Learning / AI: Python, Machine Learning, Deep Learning, NLP, TensorFlow, PyTorch, scikit-learn, pandas, numpy, SQL, AWS, Azure, Git, Docker, communication, teamwork.

    Graphic Design: Graphic Design, Adobe Creative Suite, Illustrator, Photoshop, InDesign, Figma, Framer, Procreate, web design, HTML, Microsoft Office, leadership.

    Financial Advisor: Financial Advisory, Investment, Portfolio Management, Estate Planning, Business Administration, Client Management, MS Office, ROI.
    """
    with open(job_desc_path, 'w', encoding='utf-8') as f:
        f.write(default_job)

with open(job_desc_path, 'r', encoding='utf-8') as f:
    job_desc = f.read()

job_clean = clean_text(job_desc)

# ── 5. Process Resumes (now supports .pdf, .docx, .txt) ────────────
data = []

print(f"\nScanning: {resume_folder}")
for filename in os.listdir(resume_folder):
    file_path = os.path.join(resume_folder, filename)
    if os.path.isfile(file_path):
        ext = os.path.splitext(filename)[1].lower()
        if ext in ['.pdf', '.docx', '.txt']:
            print(f"Processing: {filename}")
            raw_text = extract_resume_text(file_path)
            if raw_text:
                features = extract_features(raw_text)
                data.append({
                    'filename': filename,
                    'clean_text': clean_text(raw_text),
                    'skills': ', '.join(features['skills']),
                    'years_exp': features['years_experience'],
                    'education': features['education_level'],
                    'preview': features['raw_preview']
                })
            else:
                print(f"→ No text extracted from {filename}")

if not data:
    print("\nNo valid files processed.")
    print("Supported formats: .pdf, .docx, .txt")
    print("Folder checked:", resume_folder)
else:
    df = pd.DataFrame(data)
    print(f"\nSuccessfully processed {len(df)} files.")

    # ── 6. TF-IDF Similarity Score ─────────────────────────────────
    all_texts = [job_clean] + df['clean_text'].tolist()
    vectorizer = TfidfVectorizer(max_features=12000, ngram_range=(1, 3), stop_words='english')
    tfidf = vectorizer.fit_transform(all_texts)
    sim_scores = cosine_similarity(tfidf[0:1], tfidf[1:]).flatten() * 100

    # ── 7. Weighted Final Ranking Score ─────────────────────────────
    skill_match_weight = 0.60
    exp_weight = 0.25
    edu_weight = 0.15

    df['exp_score'] = np.clip(df['years_exp'] / 20.0, 0, 1) * 100

    edu_map = {'Phd': 100, 'Master': 85, 'Bachelor': 70, 'Unknown': 40}
    df['edu_score'] = df['education'].map(edu_map).fillna(40)

    df['final_score'] = (
        sim_scores * skill_match_weight +
        df['exp_score'] * exp_weight +
        df['edu_score'] * edu_weight
    ).round(2)

    # ── 8. Rank & Display ──────────────────────────────────────────────
    df_ranked = df.sort_values('final_score', ascending=False).reset_index(drop=True)
    df_ranked['rank'] = df_ranked.index + 1

    print("\n" + "="*100)
    print("              CANDIDATE RANKING – Top Matches")
    print("="*100)
    print(df_ranked[['rank', 'filename', 'final_score', 'skills', 'years_exp', 'education']].head(10).to_string(index=False))

    # Save results
    df_ranked.to_csv("candidate_ranking.csv", index=False)
    print(f"\nRanking saved to: {os.path.abspath('candidate_ranking.csv')}")

All dependencies loaded successfully.
Scanning: C:\Users\NARKEES SALEEM\resumes

Scanning: C:\Users\NARKEES SALEEM\resumes
Processing: aparna_graphic.docx
Processing: richard_financial.docx

Successfully processed 2 files.

              CANDIDATE RANKING – Top Matches
 rank               filename  final_score                                                                                                                                                                                                                                                                                                                                         skills  years_exp education
    1 richard_financial.docx        11.27 300+, 7+ years, 90+, advisor, ai, asset management, business administration, client satisfaction, estate planning, financial advisor, houston, investment, january 20xx, january 20xx - january 20xx, less than 2 year, less than 6 months, maverick capital, ms office, new orleans, roi, senior,

In [11]:
# FUTURE_ML_03 – Resume Screening & Candidate Ranking System
# Future Interns Machine Learning Internship – Task 3
# Author: Narkeesbanus | Chennai, Tamil Nadu | February 2026

import os
import re
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import pdfplumber
from docx import Document
import spacy
import nltk
from nltk.corpus import stopwords
from datetime import datetime

# ── Initial Setup ──────────────────────────────────────────────────
nltk.download('stopwords', quiet=True)
stop_words = set(stopwords.words('english'))

try:
    nlp = spacy.load("en_core_web_sm")
    print("spaCy model 'en_core_web_sm' loaded successfully.")
except OSError:
    print("spaCy model not found. Please run: python -m spacy download en_core_web_sm")
    raise

print("All dependencies loaded successfully.")

# ── 1. Resume Text Extractor (supports PDF, DOCX, TXT) ─────────────
def extract_resume_text(file_path):
    ext = os.path.splitext(file_path)[1].lower()
    text = ""
    
    try:
        if ext == '.pdf':
            print(f"  → Reading PDF: {file_path}")
            with pdfplumber.open(file_path) as pdf:
                for i, page in enumerate(pdf.pages, 1):
                    page_text = page.extract_text()
                    if page_text:
                        text += page_text + "\n"
                    else:
                        print(f"    Page {i}: No text extracted (possibly image-based page)")
                        
        elif ext == '.docx':
            print(f"  → Reading DOCX: {file_path}")
            doc = Document(file_path)
            for para in doc.paragraphs:
                if para.text.strip():
                    text += para.text + "\n"
                    
        elif ext in ['.txt', '.text']:
            print(f"  → Reading TXT: {file_path}")
            with open(file_path, 'r', encoding='utf-8', errors='ignore') as f:
                text = f.read()
                
        else:
            print(f"  → Unsupported file type: {ext} - {file_path}")
            return ""
            
        text = text.strip()
        print(f"    Extracted text length: {len(text)} characters")
        if len(text) > 0 and len(text) < 100:
            print(f"    Preview: {text[:100]}...")
        return text
        
    except Exception as e:
        print(f"  → Error reading {file_path}: {str(e)}")
        return ""

# ── 2. Feature Extraction (skills, experience, education) ──────────
def extract_features(text):
    if not text:
        return {'skills': [], 'years_experience': 0, 'education_level': 'Unknown', 'raw_preview': ""}
    
    doc = nlp(text.lower())
    skills = set()
    
    # Named entities
    for ent in doc.ents:
        if ent.label_ in ['ORG', 'GPE', 'DATE', 'NORP', 'PRODUCT']:
            skills.add(ent.text.strip())
    
    # Expanded skill regex (covers ML, design, financial from your resumes)
    skill_patterns = r'(python|java|c\+\+|sql|machine learning|deep learning|nlp|tensorflow|pytorch|keras|scikit-learn|pandas|numpy|aws|azure|git|docker|kubernetes|react|javascript|html|css|data science|ai|ml|big data|hadoop|spark|tableau|power bi|excel|communication|leadership|teamwork|agile|devops|financial advisor|investment|portfolio management|estate planning|business administration|graphic design|adobe creative suite|illustrator|photoshop|indesign|dreamweaver|figma|framer|procreate|ms office|microsoft office|roi|asset management|client satisfaction|brand awareness|web design|senior|junior|specialist|advisor|wells fargo|suntrust|maverick capital|experion|stepping stone|redfin)'
    found = re.findall(skill_patterns, text.lower())
    skills.update(found)
    
    # Years of experience (heuristic)
    years = 0
    years_matches = re.findall(r'\b(20\d{2})\b', text)
    if years_matches:
        try:
            earliest = min(int(y) for y in years_matches if int(y) < datetime.now().year)
            years = datetime.now().year - earliest
            years = max(0, min(years, 30))
        except:
            pass
    
    # Education level
    edu_keywords = ['phd', 'doctorate', 'master', 'mba', 'bachelor', 'bs', 'ba', 'b.tech', 'b.e', 'degree']
    edu_level = 'Unknown'
    for kw in edu_keywords:
        if kw in text.lower():
            edu_level = kw.capitalize()
            break
    
    return {
        'skills': sorted(list(set(skills))),
        'years_experience': years,
        'education_level': edu_level,
        'raw_preview': text[:400] + "..." if len(text) > 400 else text
    }

# ── 3. Clean Text for TF-IDF ──────────────────────────────────────
def clean_text(text):
    if not text:
        return ""
    text = re.sub(r'[^a-zA-Z\s]', '', text.lower())
    words = [w for w in text.split() if w not in stop_words and len(w) > 2]
    return ' '.join(words)

# ── 4. Configuration ───────────────────────────────────────────────
resume_folder = r"C:\Users\NARKEES SALEEM\resumes"  # Your confirmed path
job_desc_path = "sample_job_description.txt"

# Create job desc if missing (mixed ML/design/financial)
if not os.path.exists(job_desc_path):
    default_job = """
    Hiring for multiple roles:

    Machine Learning Engineer: Python, Machine Learning, Deep Learning, NLP, TensorFlow or PyTorch, scikit-learn, pandas, numpy, SQL, AWS or Azure, Git, Docker, communication, teamwork.

    Senior Graphic Designer: Graphic Design, Adobe Creative Suite, Illustrator, Photoshop, InDesign, Figma, Framer, Procreate, web design, HTML, Microsoft Office, leadership, client management.

    Financial Advisor: Financial Advisory, Investment, Portfolio Management, Estate Planning, Business Administration, Client Management, MS Office, ROI, communication.
    """
    with open(job_desc_path, 'w', encoding='utf-8') as f:
        f.write(default_job)
    print(f"Created sample job description: {job_desc_path}")

with open(job_desc_path, 'r', encoding='utf-8') as f:
    job_desc = f.read()

job_clean = clean_text(job_desc)

# ── 5. Process all supported files ─────────────────────────────────
data = []

print(f"\nScanning folder: {resume_folder}")
print(f"Full path: {os.path.abspath(resume_folder)}")

files = os.listdir(resume_folder)
print(f"Total files in folder: {len(files)}")
print("All files:", files)

for filename in files:
    file_path = os.path.join(resume_folder, filename)
    if os.path.isfile(file_path):
        ext = os.path.splitext(filename)[1].lower()
        if ext in ['.pdf', '.docx', '.txt']:
            print(f"\nProcessing: {filename}")
            raw_text = extract_resume_text(file_path)
            if raw_text:
                print(f"  Text extracted successfully ({len(raw_text)} characters)")
                features = extract_features(raw_text)
                data.append({
                    'filename': filename,
                    'clean_text': clean_text(raw_text),
                    'skills': ', '.join(features['skills']),
                    'years_exp': features['years_experience'],
                    'education': features['education_level'],
                    'preview': features['raw_preview']
                })
            else:
                print(f"  No text could be extracted from {filename}")
        else:
            print(f"  Skipping unsupported file type: {filename}")

# ── 6. Ranking if any files were processed ─────────────────────────
if not data:
    print("\nNo valid resumes processed.")
    print("Supported formats: .pdf, .docx, .txt")
    print("Please check:")
    print("1. Files are in:", resume_folder)
    print("2. Files are valid and contain selectable text (not scanned images)")
    print("3. Try creating .txt versions by copying text from PDFs")
else:
    df = pd.DataFrame(data)
    print(f"\nSuccessfully processed {len(df)} resumes/documents.")

    # TF-IDF similarity
    all_texts = [job_clean] + df['clean_text'].tolist()
    vectorizer = TfidfVectorizer(max_features=12000, ngram_range=(1, 3), stop_words='english')
    tfidf = vectorizer.fit_transform(all_texts)
    sim_scores = cosine_similarity(tfidf[0:1], tfidf[1:]).flatten() * 100

    # Weighted final score
    skill_match_weight = 0.60
    exp_weight = 0.25
    edu_weight = 0.15

    df['exp_score'] = np.clip(df['years_exp'] / 20.0, 0, 1) * 100
    edu_map = {'Phd': 100, 'Master': 85, 'Bachelor': 70, 'Unknown': 40}
    df['edu_score'] = df['education'].map(edu_map).fillna(40)

    df['final_score'] = (
        sim_scores * skill_match_weight +
        df['exp_score'] * exp_weight +
        df['edu_score'] * edu_weight
    ).round(2)

    # Rank & display
    df_ranked = df.sort_values('final_score', ascending=False).reset_index(drop=True)
    df_ranked['rank'] = df_ranked.index + 1

    print("\n" + "="*100)
    print("              CANDIDATE RANKING – Top Matches")
    print("="*100)
    print(df_ranked[['rank', 'filename', 'final_score', 'skills', 'years_exp', 'education']].head(10).to_string(index=False))

    # Save
    df_ranked.to_csv("candidate_ranking.csv", index=False)
    print(f"\nRanking saved to: {os.path.abspath('candidate_ranking.csv')}")

spaCy model 'en_core_web_sm' loaded successfully.
All dependencies loaded successfully.

Scanning folder: C:\Users\NARKEES SALEEM\resumes
Full path: C:\Users\NARKEES SALEEM\resumes
Total files in folder: 2
All files: ['aparna_graphic.docx', 'richard_financial.docx']

Processing: aparna_graphic.docx
  → Reading DOCX: C:\Users\NARKEES SALEEM\resumes\aparna_graphic.docx
    Extracted text length: 1318 characters
  Text extracted successfully (1318 characters)

Processing: richard_financial.docx
  → Reading DOCX: C:\Users\NARKEES SALEEM\resumes\richard_financial.docx
    Extracted text length: 1877 characters
  Text extracted successfully (1877 characters)

Successfully processed 2 resumes/documents.

              CANDIDATE RANKING – Top Matches
 rank               filename  final_score                                                                                                                                                                                                              

In [12]:
# FUTURE_ML_03 – Resume Screening & Candidate Ranking System
# Future Interns Machine Learning Internship – Task 3
# Author: Narkeesbanus | Chennai, Tamil Nadu | February 2026

import os
import re
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import pdfplumber
from docx import Document
import spacy
import nltk
from nltk.corpus import stopwords
from datetime import datetime

# ── Initial Setup ──────────────────────────────────────────────────
nltk.download('stopwords', quiet=True)
stop_words = set(stopwords.words('english'))

try:
    nlp = spacy.load("en_core_web_sm")
    print("spaCy model loaded successfully.")
except OSError as e:
    print("spaCy model not found:", e)
    print("Run: python -m spacy download en_core_web_sm")
    raise

print("All dependencies loaded successfully.")

# ── 1. Resume Text Extractor (PDF + DOCX + TXT) ────────────────────
def extract_resume_text(file_path):
    ext = os.path.splitext(file_path)[1].lower()
    text = ""
    
    print(f"\nReading file: {file_path}")
    
    try:
        if ext == '.pdf':
            with pdfplumber.open(file_path) as pdf:
                page_count = len(pdf.pages)
                print(f"  PDF pages: {page_count}")
                for i, page in enumerate(pdf.pages, 1):
                    page_text = page.extract_text()
                    if page_text:
                        text += page_text + "\n"
                    else:
                        print(f"    Page {i}: No text found (image-based?)")
                        
        elif ext == '.docx':
            doc = Document(file_path)
            para_count = len(doc.paragraphs)
            print(f"  DOCX paragraphs: {para_count}")
            for para in doc.paragraphs:
                if para.text.strip():
                    text += para.text + "\n"
                    
        elif ext in ['.txt', '.text']:
            with open(file_path, 'r', encoding='utf-8', errors='ignore') as f:
                text = f.read()
            print(f"  TXT file read ({len(text)} chars)")
            
        else:
            print(f"  Unsupported extension: {ext}")
            return ""
            
        text = text.strip()
        print(f"  Extracted text length: {len(text)} characters")
        if text:
            print(f"  Preview (first 200 chars): {text[:200]}...")
        else:
            print("  WARNING: No text was extracted from this file")
        return text
        
    except Exception as e:
        print(f"  ERROR reading file: {str(e)}")
        return ""

# ── 2. Feature Extraction ──────────────────────────────────────────
def extract_features(text):
    if not text:
        return {'skills': [], 'years_experience': 0, 'education_level': 'Unknown', 'raw_preview': ""}
    
    doc = nlp(text.lower())
    skills = set()
    
    # Named entities
    for ent in doc.ents:
        if ent.label_ in ['ORG', 'GPE', 'DATE', 'NORP', 'PRODUCT']:
            skills.add(ent.text.strip())
    
    # Skill/keyword regex (covers your two resumes + common ML/design terms)
    skill_patterns = r'(python|java|c\+\+|sql|machine learning|deep learning|nlp|tensorflow|pytorch|keras|scikit-learn|pandas|numpy|aws|azure|git|docker|kubernetes|react|javascript|html|css|data science|ai|ml|big data|hadoop|spark|tableau|power bi|excel|communication|leadership|teamwork|agile|devops|financial advisor|investment|portfolio management|estate planning|business administration|graphic design|adobe creative suite|illustrator|photoshop|indesign|dreamweaver|figma|framer|procreate|ms office|microsoft office|roi|asset management|client satisfaction|brand awareness|web design|senior|junior|specialist|advisor|wells fargo|suntrust|maverick capital|experion|stepping stone|redfin)'
    found = re.findall(skill_patterns, text.lower())
    skills.update(found)
    
    # Experience years (simple heuristic)
    years = 0
    years_matches = re.findall(r'\b(20\d{2})\b', text)
    if years_matches:
        try:
            earliest = min(int(y) for y in years_matches if int(y) < datetime.now().year)
            years = datetime.now().year - earliest
            years = max(0, min(years, 30))
        except:
            pass
    
    # Education
    edu_keywords = ['phd', 'doctorate', 'master', 'mba', 'bachelor', 'bs', 'ba', 'b.tech', 'b.e', 'degree']
    edu_level = 'Unknown'
    for kw in edu_keywords:
        if kw in text.lower():
            edu_level = kw.capitalize()
            break
    
    return {
        'skills': sorted(list(set(skills))),
        'years_experience': years,
        'education_level': edu_level,
        'raw_preview': text[:400] + "..." if len(text) > 400 else text
    }

# ── 3. Clean Text ──────────────────────────────────────────────────
def clean_text(text):
    if not text:
        return ""
    text = re.sub(r'[^a-zA-Z\s]', '', text.lower())
    words = [w for w in text.split() if w not in stop_words and len(w) > 2]
    return ' '.join(words)

# ── 4. Configuration ───────────────────────────────────────────────
resume_folder = r"C:\Users\NARKEES SALEEM\resumes"  # your confirmed path
job_desc_path = "sample_job_description.txt"

# Create job desc if missing
if not os.path.exists(job_desc_path):
    default_job = """
    Hiring for multiple roles:

    Machine Learning Engineer: Python, Machine Learning, Deep Learning, NLP, TensorFlow or PyTorch, scikit-learn, pandas, numpy, SQL, AWS or Azure, Git, Docker, communication, teamwork.

    Senior Graphic Designer: Graphic Design, Adobe Creative Suite, Illustrator, Photoshop, InDesign, Figma, Framer, Procreate, web design, HTML, Microsoft Office, leadership, client management.

    Financial Advisor: Financial Advisory, Investment, Portfolio Management, Estate Planning, Business Administration, Client Management, MS Office, ROI, communication.
    """
    with open(job_desc_path, 'w', encoding='utf-8') as f:
        f.write(default_job)
    print(f"Created sample job description: {job_desc_path}")

with open(job_desc_path, 'r', encoding='utf-8') as f:
    job_desc = f.read()

job_clean = clean_text(job_desc)

# ── 5. Process Files ───────────────────────────────────────────────
data = []

print(f"\nScanning folder: {resume_folder}")
print(f"Full path: {os.path.abspath(resume_folder)}")

files = os.listdir(resume_folder)
print(f"Total files: {len(files)}")
print("Files:", files)

for filename in files:
    file_path = os.path.join(resume_folder, filename)
    if os.path.isfile(file_path):
        ext = os.path.splitext(filename)[1].lower()
        if ext in ['.pdf', '.docx', '.txt']:
            print(f"\n=== Processing: {filename} ===")
            raw_text = extract_resume_text(file_path)
            if raw_text:
                features = extract_features(raw_text)
                data.append({
                    'filename': filename,
                    'clean_text': clean_text(raw_text),
                    'skills': ', '.join(features['skills']),
                    'years_exp': features['years_experience'],
                    'education': features['education_level'],
                    'preview': features['raw_preview']
                })
            else:
                print("   → No text extracted (likely image-based PDF or empty file)")
        else:
            print(f"   Skipping unsupported file: {filename}")

# ── 6. Ranking ─────────────────────────────────────────────────────
if not data:
    print("\nNo valid documents processed.")
    print("Please check:")
    print("1. Files are in:", resume_folder)
    print("2. Files are .pdf, .docx or .txt")
    print("3. Files contain selectable text (not scanned images)")
    print("   Try opening in Word/Reader and copy-paste to .txt if needed")
else:
    df = pd.DataFrame(data)
    print(f"\nSuccessfully processed {len(df)} documents.")

    # TF-IDF similarity
    all_texts = [job_clean] + df['clean_text'].tolist()
    vectorizer = TfidfVectorizer(max_features=12000, ngram_range=(1, 3), stop_words='english')
    tfidf = vectorizer.fit_transform(all_texts)
    sim_scores = cosine_similarity(tfidf[0:1], tfidf[1:]).flatten() * 100

    # Final weighted score
    skill_weight = 0.60
    exp_weight = 0.25
    edu_weight = 0.15

    df['exp_score'] = np.clip(df['years_exp'] / 20.0, 0, 1) * 100

    edu_map = {'Phd': 100, 'Master': 85, 'Bachelor': 70, 'Unknown': 40}
    df['edu_score'] = df['education'].map(edu_map).fillna(40)

    df['final_score'] = (
        sim_scores * skill_weight +
        df['exp_score'] * exp_weight +
        df['edu_score'] * edu_weight
    ).round(2)

    # Rank & display
    df_ranked = df.sort_values('final_score', ascending=False).reset_index(drop=True)
    df_ranked['rank'] = df_ranked.index + 1

    print("\n" + "="*90)
    print("              CANDIDATE RANKING – Top Matches")
    print("="*90)
    print(df_ranked[['rank', 'filename', 'final_score', 'skills', 'years_exp', 'education']].head(10).to_string(index=False))

    # Save
    output_file = "candidate_ranking.csv"
    df_ranked.to_csv(output_file, index=False)
    print(f"\nRanking saved to: {os.path.abspath(output_file)}")

spaCy model loaded successfully.
All dependencies loaded successfully.

Scanning folder: C:\Users\NARKEES SALEEM\resumes
Full path: C:\Users\NARKEES SALEEM\resumes
Total files: 2
Files: ['aparna_graphic.docx', 'richard_financial.docx']

=== Processing: aparna_graphic.docx ===

Reading file: C:\Users\NARKEES SALEEM\resumes\aparna_graphic.docx
  DOCX paragraphs: 24
  Extracted text length: 1318 characters
  Preview (first 200 chars): Professional Experience
JAN `XX-PRESENT
Senior Graphic Design Specialist I Experion, New York, NY
Lead the design, development and implementation of graphic, layout, and production communication mater...

=== Processing: richard_financial.docx ===

Reading file: C:\Users\NARKEES SALEEM\resumes\richard_financial.docx
  DOCX paragraphs: 22
  Extracted text length: 1877 characters
  Preview (first 200 chars): Financial Advisor with 7+ years of experience delivering financial/investment advisory services to high value clients. Proven success in managing multi-mi